# Day 2: Baseline and machine learning models

**Research question 1:** Can a firm's characteristics predict which energy pathway it is on (no renewable, purchased, or on-site generation)? Does machine learning beat multinomial logistic regression?

This notebook:

1. Loads the clean dataset from Day 1.
2. Sets aside 20% of firms as a **test set** that no model sees during training.
3. Trains three models on three sets of predictors.
4. Scores every model on the test set and reports how certain each score is.

**How to run it:** click *Runtime > Run all*. When asked, upload `sme_energy_clean.csv` (the file Day 1 produced). It takes about five minutes, because it trains several hundred models.

## 0. Setup

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (roc_auc_score, f1_score, balanced_accuracy_score, accuracy_score, log_loss,
                             confusion_matrix, recall_score)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
SEED = 42          # fixes the random choices, so you and I get the same numbers

DATA_FILE = "sme_energy_clean.csv"
if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files
        print("Choose sme_energy_clean.csv (the file Day 1 produced)")
        DATA_FILE = list(files.upload().keys())[0]
    except ImportError:
        raise FileNotFoundError("Put sme_energy_clean.csv in the same folder as this notebook.")

df = pd.read_csv(DATA_FILE)
print("Firms:", len(df), "| Columns:", df.shape[1])

## 1. Outcome and predictors

The outcome has three classes. They are not equal in size, which matters later.

In [ ]:
CLASS_NAMES = ["No renewable", "Purchased", "On-site"]      # codes 0, 1, 2
y = df["pathway_code"]
pd.DataFrame({"Firms": y.value_counts().sort_index().values,
              "Share %": (100 * y.value_counts(normalize=True).sort_index().values).round(1)}, index=CLASS_NAMES)

Three predictor sets, each adding to the one before. Comparing them shows *where* the predictive information comes from.

| Set | Contains | Question it answers |
|---|---|---|
| **A1 Firm** | Sector, size, age, turnover, growth, hiring | How far do firm characteristics alone get us? |
| **A2 Firm + country** | A1 plus the country | How much does national context add? |
| **B All** | A2 plus what the firm does: other green actions, investment, barriers, support, climate strategy | How much more do behaviours add? |

Left out on purpose:

- `region` and `sector`, because `country` and `nace_section` already contain them in finer detail.
- `plans_renewable` and `onsite_share`, because they are about the outcome itself (data leakage).

In [ ]:
# Two skewed numbers are put on a log scale so that very large firms do not dominate.
df["log_employees"] = np.log1p(df["employees"])
df["log_firm_age"] = np.log1p(df["firm_age_years"].clip(lower=0))

FIRM = ["nace_section", "size", "log_employees", "log_firm_age", "age_band", "turnover_band",
        "turnover_trend", "employment_trend", "hiring_difficulty", "sells"]
COUNTRY = ["country"]
BEHAVIOUR = ([c for c in df.columns if c.startswith(("act_", "sup_", "ext_", "bar_", "help_"))]
             + ["n_other_actions", "n_barriers", "asked_followups", "investment", "climate_strategy",
                "green_products", "green_jobs", "recycled_materials", "lifespan_policy"])

FEATURE_SETS = {"A1 Firm": FIRM,
                "A2 Firm + country": FIRM + COUNTRY,
                "B All": FIRM + COUNTRY + BEHAVIOUR}
for name, cols in FEATURE_SETS.items():
    print(f"{name:<20} {len(cols)} predictors")

## 2. Hold back a test set

20% of firms are set aside now and used only once, at the end. A model's score on firms it has never seen is the honest measure of how well it predicts. `stratify` keeps the three classes in the same proportions in both parts.

In [ ]:
train_idx, test_idx = train_test_split(df.index, test_size=0.20, stratify=y, random_state=SEED)
y_train, y_test = y.loc[train_idx], y.loc[test_idx]
print("Training firms:", len(train_idx), "| Test firms:", len(test_idx))
print("Class shares in test set %:", (100 * y_test.value_counts(normalize=True).sort_index()).round(1).tolist())

## 3. Preparing the predictors

Models need numbers, so:

- **Categories** (such as country) become one yes/no column per category. This is called one-hot encoding.
- **Missing numbers** (employees, firm age) are filled with the median, and an extra column records that the value was missing.
- **Numbers** are put on a common scale, which logistic regression needs.

This is wrapped in a `Pipeline` so that it is learned from the training firms only and then applied to the test firms. Learning it from all firms would leak information from the test set.

In [ ]:
NUM_WITH_GAPS = ["log_employees", "log_firm_age"]

def make_preprocessor(columns):
    cat = [c for c in columns if not pd.api.types.is_numeric_dtype(df[c])]
    gaps = [c for c in columns if c in NUM_WITH_GAPS]
    num = [c for c in columns if pd.api.types.is_numeric_dtype(df[c]) and c not in NUM_WITH_GAPS]
    steps = [("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat),
             ("gaps", Pipeline([("fill", SimpleImputer(strategy="median", add_indicator=True)),
                                ("scale", StandardScaler())]), gaps)]
    if num:
        steps.append(("num", StandardScaler(), num))
    return ColumnTransformer(steps)

example = make_preprocessor(FEATURE_SETS["B All"]).fit(df.loc[train_idx, FEATURE_SETS["B All"]])
print("Set B becomes", example.transform(df.loc[train_idx[:5], FEATURE_SETS["B All"]]).shape[1], "numeric columns after encoding")

## 4. The models

| Model | What it is | Role |
|---|---|---|
| **Logistic regression** (multinomial) | The standard statistical model for a categorical outcome. Assumes each predictor adds a fixed amount to the log-odds | The baseline to beat |
| **Random forest** | Hundreds of decision trees, each trained on a random part of the data, voting together | Machine learning |
| **XGBoost** | Trees built one after another, each correcting the errors of the ones before | Machine learning |

**Class imbalance.** About six in ten firms use no renewable energy. A model could score about 60% accuracy by always answering "no renewable", which is useless. All three models are therefore told to treat the classes as equally important (`balanced` weights), and we judge them on measures that are not fooled by imbalance.

**Tuning.** Each model has settings that are not learned from data (for example, how deep a tree can grow). We try several and keep the best, judged by **5-fold cross-validation**: the training firms are split into five parts, the model trains on four and is scored on the fifth, five times over.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def build_search(model_name, columns):
    pre = make_preprocessor(columns)
    if model_name == "Logistic regression":
        pipe = Pipeline([("pre", pre), ("model", LogisticRegression(max_iter=3000, class_weight="balanced"))])
        return GridSearchCV(pipe, {"model__C": [0.01, 0.1, 1.0]}, scoring="roc_auc_ovr", cv=cv, n_jobs=-1)
    if model_name == "Random forest":
        pipe = Pipeline([("pre", pre), ("model", RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample",
                                                                       n_jobs=-1, random_state=SEED))])
        return GridSearchCV(pipe, {"model__min_samples_leaf": [3, 10], "model__max_features": ["sqrt", 0.3]},
                            scoring="roc_auc_ovr", cv=cv, n_jobs=1)
    if model_name == "XGBoost":
        pipe = Pipeline([("pre", pre), ("model", XGBClassifier(objective="multi:softprob", tree_method="hist",
                                                               n_jobs=2, random_state=SEED, verbosity=0))])
        grid = {"model__max_depth": [3, 4, 6], "model__learning_rate": [0.03, 0.1], "model__n_estimators": [200, 400],
                "model__colsample_bytree": [0.6, 0.9], "model__min_child_weight": [1, 5], "model__subsample": [0.8]}
        return RandomizedSearchCV(pipe, grid, n_iter=8, scoring="roc_auc_ovr", cv=cv, n_jobs=1, random_state=SEED)

MODELS = ["Logistic regression", "Random forest", "XGBoost"]

## 5. Train everything

Nine combinations: three models on three predictor sets. This is the slow cell.

In [ ]:
fitted, rows = {}, []
balanced_w = compute_sample_weight("balanced", y_train)        # XGBoost takes the class weights this way
start = time.time()
for set_name, columns in FEATURE_SETS.items():
    X_train = df.loc[train_idx, columns]
    for model_name in MODELS:
        t0 = time.time()
        search = build_search(model_name, columns)
        if model_name == "XGBoost":
            search.fit(X_train, y_train, model__sample_weight=balanced_w)
        else:
            search.fit(X_train, y_train)
        fitted[(set_name, model_name)] = search.best_estimator_
        best = search.best_index_
        rows.append({"Predictor set": set_name, "Model": model_name,
                     "CV AUC (mean)": round(search.cv_results_["mean_test_score"][best], 3),
                     "CV AUC (sd)": round(search.cv_results_["std_test_score"][best], 3),
                     "Best settings": str({k.replace("model__", ""): v for k, v in search.best_params_.items()})})
        print(f"{set_name:<18} {model_name:<20} CV AUC {rows[-1]['CV AUC (mean)']:.3f}   ({time.time() - t0:.0f}s)")
print(f"\nTotal training time: {(time.time() - start) / 60:.1f} minutes")
cv_table = pd.DataFrame(rows)

## 6. Score on the test set

How to read the measures:

| Measure | Meaning | Chance level |
|---|---|---|
| **AUC** | Given one firm from a class and one firm not from it, how often the model ranks them correctly. Averaged over the three classes. Our main measure | 0.50 |
| **Balanced accuracy** | Average of the share correctly identified in each class | 0.33 |
| **Macro F1** | Balances catching firms of a class against false alarms, averaged over classes | about 0.33 |
| **Accuracy** | Share of firms classified correctly. Reported for completeness; misleading with unequal classes | about 0.61 (always guessing "no renewable") |

In [ ]:
def score(model, columns):
    X_test = df.loc[test_idx, columns]
    proba = model.predict_proba(X_test)
    pred = proba.argmax(axis=1)
    return proba, {"AUC": roc_auc_score(y_test, proba, multi_class="ovr"),
                   "Balanced accuracy": balanced_accuracy_score(y_test, pred),
                   "Macro F1": f1_score(y_test, pred, average="macro"),
                   "Accuracy": accuracy_score(y_test, pred),
                   "Log loss": log_loss(y_test, proba)}

probas, rows = {}, []
for (set_name, model_name), model in fitted.items():
    probas[(set_name, model_name)], s = score(model, FEATURE_SETS[set_name])
    rows.append({"Predictor set": set_name, "Model": model_name, **{k: round(v, 3) for k, v in s.items()}})

# Reference point: a "model" that ignores the predictors and always predicts the overall class shares.
dummy = DummyClassifier(strategy="prior").fit(df.loc[train_idx, FIRM], y_train)
dp = dummy.predict_proba(df.loc[test_idx, FIRM])
rows.append({"Predictor set": "(none)", "Model": "No predictors", "AUC": 0.5,
             "Balanced accuracy": round(balanced_accuracy_score(y_test, dp.argmax(axis=1)), 3),
             "Macro F1": round(f1_score(y_test, dp.argmax(axis=1), average="macro"), 3),
             "Accuracy": round(accuracy_score(y_test, dp.argmax(axis=1)), 3), "Log loss": round(log_loss(y_test, dp), 3)})
test_table = pd.DataFrame(rows)
test_table

### How certain are these scores?

The test set is one sample of about 2,000 firms. A different sample would give slightly different scores. The **bootstrap** estimates that uncertainty: we redraw the test firms at random 1,000 times and recompute the AUC each time. The middle 95% of those values is the confidence interval.

The second table answers research question 1 directly: the difference between XGBoost and logistic regression, with its interval. If the interval excludes zero, the difference is statistically reliable. Whether it is large enough to matter is a separate question: look at the size of the difference itself.

In [ ]:
rng = np.random.default_rng(SEED)
yt = y_test.values
boot_idx = [rng.integers(0, len(yt), len(yt)) for _ in range(1000)]

def boot_auc(proba):
    return np.array([roc_auc_score(yt[i], proba[i], multi_class="ovr") for i in boot_idx])

boots = {key: boot_auc(p) for key, p in probas.items()}
ci = pd.DataFrame([{"Predictor set": k[0], "Model": k[1],
                    "AUC": round(roc_auc_score(yt, probas[k], multi_class="ovr"), 3),
                    "95% CI low": round(np.percentile(b, 2.5), 3), "95% CI high": round(np.percentile(b, 97.5), 3)}
                   for k, b in boots.items()])

diff_rows = []
for set_name in FEATURE_SETS:
    for ml in ["Random forest", "XGBoost"]:
        dvals = boots[(set_name, ml)] - boots[(set_name, "Logistic regression")]
        point = (roc_auc_score(yt, probas[(set_name, ml)], multi_class="ovr")
                 - roc_auc_score(yt, probas[(set_name, "Logistic regression")], multi_class="ovr"))
        lo, hi = np.percentile(dvals, [2.5, 97.5])
        diff_rows.append({"Predictor set": set_name, "Comparison": f"{ml} minus logistic regression",
                          "AUC difference": point, "95% CI low": lo, "95% CI high": hi,
                          "CI excludes zero?": "Yes" if (lo > 0 or hi < 0) else "No"})
diff_table = pd.DataFrame(diff_rows)          # kept unrounded for the results file; rounded only for display
display(ci); display(diff_table.round(4))

### Which classes are easy and which are hard?

In [ ]:
# The model we go on to interpret is chosen by its cross-validated score on the TRAINING firms, not by its test score.
# Choosing on the test set would use the test firms twice.
best_row = cv_table.loc[cv_table["CV AUC (mean)"].idxmax()]
best_key = (best_row["Predictor set"], best_row["Model"])
test_best = max(probas, key=lambda k: roc_auc_score(yt, probas[k], multi_class="ovr"))
print("Highest cross-validated AUC:", best_key[1], "with predictor set", best_key[0], "(this is the model interpreted on Day 3)")
print("Highest test AUC:           ", test_best[1], "with predictor set", test_best[0])

rows = []
for key, p in probas.items():
    row = {"Predictor set": key[0], "Model": key[1]}
    for c, name in enumerate(CLASS_NAMES):
        row[f"AUC: {name}"] = round(roc_auc_score((yt == c).astype(int), p[:, c]), 3)
    rec = recall_score(yt, p.argmax(axis=1), average=None)
    for c, name in enumerate(CLASS_NAMES):
        row[f"Recall: {name}"] = round(rec[c], 3)
    rows.append(row)
per_class = pd.DataFrame(rows)
per_class

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
pred_best = probas[best_key].argmax(axis=1)
cm = confusion_matrix(yt, pred_best)
cm_counts = pd.DataFrame(cm, index=[f"Actual: {n}" for n in CLASS_NAMES], columns=[f"Predicted: {n}" for n in CLASS_NAMES])
prec, rec, f1, sup = precision_recall_fscore_support(yt, pred_best)
best_detail = pd.DataFrame({"Test firms": sup, "Recall": rec, "Precision": prec, "F1": f1,
                            "AUC (one-vs-rest)": [roc_auc_score((yt == c).astype(int), probas[best_key][:, c]) for c in range(3)]}, index=CLASS_NAMES).round(4)
# Confidence intervals for the two classification measures of the chosen model, from the same bootstrap resamples
bs = np.array([(balanced_accuracy_score(yt[i], pred_best[i]), f1_score(yt[i], pred_best[i], average="macro"), accuracy_score(yt[i], pred_best[i])) for i in boot_idx])
best_ci = pd.DataFrame({"Value": [balanced_accuracy_score(yt, pred_best), f1_score(yt, pred_best, average="macro"), accuracy_score(yt, pred_best)],
                        "95% CI low": np.percentile(bs, 2.5, axis=0), "95% CI high": np.percentile(bs, 97.5, axis=0)},
                       index=["Balanced accuracy", "Macro F1", "Accuracy"])
majority = pd.Series(yt).value_counts(normalize=True).max()
print("Accuracy from always predicting the largest class: %.3f" % majority)
display(cm_counts); display(best_detail); best_ci.round(3)

### Figure 3: test AUC by model and predictor set

In [ ]:
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
COLOURS = {"Logistic regression": "#2a78d6", "Random forest": "#eb6834", "XGBoost": "#1baf7a"}
MARKERS = {"Logistic regression": "o", "Random forest": "s", "XGBoost": "D"}
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": GRID, "text.color": INK,
                     "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": INK})

fig, ax = plt.subplots(figsize=(7, 3.7))
set_names = list(FEATURE_SETS)
offsets = {"Logistic regression": 0.22, "Random forest": 0.0, "XGBoost": -0.22}
for m in MODELS:
    sub = ci[ci["Model"] == m].set_index("Predictor set").loc[set_names]
    ypos = np.arange(len(set_names))[::-1] + offsets[m]
    ax.errorbar(sub["AUC"], ypos, xerr=[sub["AUC"] - sub["95% CI low"], sub["95% CI high"] - sub["AUC"]],
                fmt=MARKERS[m], color=COLOURS[m], markersize=6, capsize=3, linewidth=1.6, label=m)
ax.axvline(0.5, color=MUTED, linewidth=1, linestyle=":")
ax.text(0.502, len(set_names) - 0.45, "chance", fontsize=8, color=MUTED)
ax.set_yticks(np.arange(len(set_names))[::-1]); ax.set_yticklabels(set_names)
ax.set_ylim(-0.6, len(set_names) - 0.3)
ax.set_xlabel("AUC on the test set, with 95% confidence interval")
ax.xaxis.grid(True, color=GRID, linewidth=0.8); ax.set_axisbelow(True)
for side in ["top", "right", "left"]: ax.spines[side].set_visible(False)
ax.tick_params(length=0)
ax.legend(frameon=False, loc="lower left", bbox_to_anchor=(-0.01, 1.0), ncol=3, fontsize=8, handletextpad=0.3)
ax.set_title("Predicting an SME's energy pathway", loc="left", fontsize=11, fontweight="bold", pad=26)
plt.tight_layout(); plt.savefig("fig3_model_comparison.png", dpi=300); plt.show()

### Figure 4: where the best model is right and wrong

Each row is one actual pathway and adds up to 100%. The diagonal is the share correctly identified (the recall).

In [ ]:
cm_pct = 100 * cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(4.6, 3.8))
im = ax.imshow(cm_pct, cmap="Blues", vmin=0, vmax=100)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{cm_pct[i, j]:.0f}%\n({cm[i, j]})", ha="center", va="center", fontsize=9,
                color="white" if cm_pct[i, j] > 55 else INK)
ax.set_xticks(range(3)); ax.set_xticklabels(CLASS_NAMES); ax.set_yticks(range(3)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Predicted pathway"); ax.set_ylabel("Actual pathway")
for side in ax.spines.values(): side.set_visible(False)
ax.tick_params(length=0)
ax.set_title(f"{best_key[1]}, predictor set {best_key[0]}", loc="left", fontsize=10, fontweight="bold")
plt.tight_layout(); plt.savefig("fig4_confusion_matrix.png", dpi=300); plt.show()

## 7. YOUR TURN

This cell removes predictors of your choice and shows how much the logistic regression's cross-validated AUC drops. A big drop means those predictors carried information nothing else could replace. It runs in a few seconds.

Try, one at a time: `["country"]`, `["size", "log_employees"]`, `["turnover_band"]`, `["climate_strategy"]`, `["investment"]`, and all the barriers at once with `[c for c in BEHAVIOUR if c.startswith("bar_")] + ["n_barriers"]`.

Write down the drop for each. Which single predictor matters most?

In [ ]:
REMOVE = ["country"]          # <-- change this and re-run

def cv_auc(columns):
    pipe = Pipeline([("pre", make_preprocessor(columns)),
                     ("model", LogisticRegression(max_iter=3000, class_weight="balanced", C=0.1))])
    return cross_val_score(pipe, df.loc[train_idx, columns], y_train, scoring="roc_auc_ovr", cv=cv, n_jobs=-1).mean()

full = FEATURE_SETS["B All"]
reduced = [c for c in full if c not in REMOVE]
a_full, a_reduced = cv_auc(full), cv_auc(reduced)
print(f"All predictors:          AUC {a_full:.3f}")
print(f"Without {REMOVE}: AUC {a_reduced:.3f}")
print(f"Drop:                    {a_full - a_reduced:.3f}")

## 8. Save the outputs

- `Day2_results.xlsx`: every table above, one per sheet.
- `test_predictions.csv`: for each test firm, the actual pathway and the best model's predicted probabilities.
- `fig3_model_comparison.png`, `fig4_confusion_matrix.png`.

In [ ]:
pred = pd.DataFrame(probas[best_key], columns=[f"prob_{n}" for n in CLASS_NAMES])
pred.insert(0, "firm_id", df.loc[test_idx, "firm_id"].values)
pred.insert(1, "actual", [CLASS_NAMES[c] for c in yt])
pred.insert(2, "predicted", [CLASS_NAMES[c] for c in probas[best_key].argmax(axis=1)])
pred.to_csv("test_predictions.csv", index=False)

with pd.ExcelWriter("Day2_results.xlsx") as xl:
    test_table.to_excel(xl, sheet_name="Test scores", index=False)
    ci.to_excel(xl, sheet_name="AUC confidence intervals", index=False)
    diff_table.to_excel(xl, sheet_name="ML vs logistic", index=False)
    per_class.to_excel(xl, sheet_name="Per class", index=False)
    cm_counts.to_excel(xl, sheet_name="Confusion matrix")
    best_detail.to_excel(xl, sheet_name="Chosen model per class")
    best_ci.to_excel(xl, sheet_name="Chosen model intervals")
    pd.DataFrame({"Item": ["Chosen model", "Predictor set", "Training firms", "Test firms", "Encoded columns (Set B)", "Majority-class accuracy"],
                  "Value": [best_key[1], best_key[0], len(train_idx), len(test_idx), example.transform(df.loc[train_idx[:5], FEATURE_SETS["B All"]]).shape[1], round(majority, 3)]}).to_excel(xl, sheet_name="Summary", index=False)
    cv_table.to_excel(xl, sheet_name="Cross-validation", index=False)
print("Saved.")

try:
    from google.colab import files
    for f in ["Day2_results.xlsx", "test_predictions.csv", "fig3_model_comparison.png", "fig4_confusion_matrix.png"]:
        files.download(f)
except ImportError:
    pass